# Radon Analysis

Evaluating mitigation system effectiveness from hourly radon sensor readings.

**Reference levels (pCi/L):**
- **4.0** — EPA action level (mitigate above this)
- **2.7** — WHO recommended action level
- **1.3** — average US indoor radon level

**Timeline:**
- 2026-05-12 — mitigation fan turned on
- 2026-05-14 — sensor moved to current location (readings before this are *not* comparable — sensor was previously in a lower-radon room)
- 2026-05-18 — improved sealing; vacuum increased from 2.5" to 3.5" WC
- 2026-05-23 — external suction-point adjustments (no internal changes)

**Data quality notes:**

1. **Seasonality.** This data covers late spring only. Radon levels in this climate are typically highest in winter (Nov–Mar) due to reduced ventilation and the stack effect actively pulling soil gas through the foundation. Late-spring readings represent a seasonal floor, not the worst case — year-round effectiveness can only be confirmed by observing the system through at least one full heating season.

2. **Window-opening during elevated readings.** When elevated levels were observed in real time, windows were opened to ventilate. This likely lowered some of the hourly readings during those events, so the spike peaks, durations, and counts shown below are likely *underestimates* of the unmitigated indoor levels.

3. **Sensor relocation.** Because the sensor moved on 2026-05-14, all aggregates and statistics below are restricted to that date forward. Pre-move readings appear (shaded) in the raw hourly time-series for context only.

**Framing — mean vs. distribution.** A mitigation system's mechanical function is to prevent individual excursions of soil gas, not only to lower the average. Analysis below uses percentiles, hours-above-threshold, and individual spike events alongside averages, because a system whose mean is below 4.0 but whose upper percentiles or daily max still hit EPA is reducing typical exposure without fully controlling source-gas entry.

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

EPA_LEVEL = 4.0
WHO_LEVEL = 2.7

SENSOR_MOVED = '2026-05-14'  # all analysis restricted to this date forward

# (date, short label, color) — mitigation events in green, sensor event in purple
EVENTS = [
    ('2026-05-12', 'Fan',      'green'),
    ('2026-05-14', 'Sensor →', 'purple'),
    ('2026-05-18', '3.5" WC',  'green'),
    ('2026-05-23', 'Ext.',     'green'),
]

# Phases for comparison (start inclusive, end exclusive; end=None means open-ended)
PHASES = [
    ('P1: fan + 2.5" WC',    '2026-05-14', '2026-05-18'),
    ('P2: 3.5" WC + sealing', '2026-05-18', '2026-05-23'),
    ('P3: + ext. tweaks',     '2026-05-23', None),
]

## Load data

In [2]:
df = pd.read_csv(
    '03-01-2026_06-02-2026-export.csv',
    parse_dates=['Date'],
    date_format='%m/%d/%Y %H:%M',
)
df = df.rename(columns={'Date': 'ts', 'Radon Level': 'radon'}).set_index('ts').sort_index()

df_post = df.loc[SENSOR_MOVED:].copy()  # comparable readings only

print(f'Full dataset:     {len(df):>5} rows  |  {df.index.min()} → {df.index.max()}')
print(f'Post-sensor-move: {len(df_post):>5} rows  |  {df_post.index.min()} → {df_post.index.max()}')
df.head()

Full dataset:       732 rows  |  2026-04-25 19:24:00 → 2026-06-02 20:31:00
Post-sensor-move:   346 rows  |  2026-05-14 00:25:00 → 2026-06-02 20:31:00


,radon
ts,
2026-04-25 19:24:00,1.2
2026-04-25 20:24:00,0.9
2026-04-25 21:24:00,0.4
2026-04-25 22:24:00,0.5
2026-04-25 23:24:00,0.4


## Helper: chart annotations

Uses `add_shape` + `add_annotation` directly rather than `add_hline/vline/vrect`'s `annotation_text=` parameter, to sidestep a plotly bug that trips on pandas 3.0 Timestamp arithmetic.

In [3]:
EVENTS_POST = [(d, l, c) for d, l, c in EVENTS if pd.Timestamp(d) >= pd.Timestamp(SENSOR_MOVED)]


def annotate(fig, *, levels=True, events=None, shade_pre_move=False, height=600):
    """Add EPA/WHO horizontal lines, event vertical lines, and optional pre-move shading."""
    fig.update_layout(height=height)
    if levels:
        for y, color, dash, name in [(EPA_LEVEL, 'red', 'dash', 'EPA'),
                                     (WHO_LEVEL, 'orange', 'dot', 'WHO')]:
            fig.add_shape(type='line', xref='x domain', yref='y',
                          x0=0, x1=1, y0=y, y1=y,
                          line=dict(color=color, dash=dash, width=1))
            fig.add_annotation(xref='x domain', yref='y',
                               x=1, y=y, text=f'{name} {y}', showarrow=False,
                               xanchor='right', yanchor='bottom',
                               font=dict(size=10, color=color))
    if events:
        for i, (date, label, color) in enumerate(events):
            x = pd.Timestamp(date)
            fig.add_shape(type='line', xref='x', yref='y domain',
                          x0=x, x1=x, y0=0, y1=1,
                          line=dict(color=color, dash='dash', width=1.5))
            on_top = (i % 2 == 0)
            fig.add_annotation(xref='x', yref='y domain',
                               x=x, y=1.02 if on_top else -0.04,
                               text=label, showarrow=False,
                               xanchor='center',
                               yanchor='bottom' if on_top else 'top',
                               font=dict(size=10, color=color))
    if shade_pre_move:
        fig.add_shape(type='rect', xref='x', yref='y domain',
                      x0=df.index.min(), x1=pd.Timestamp(SENSOR_MOVED),
                      y0=0, y1=1,
                      fillcolor='gray', opacity=0.18, line_width=0, layer='below')
        fig.add_annotation(xref='x', yref='y domain',
                           x=df.index.min(), y=0.97,
                           text='not comparable (sensor in other room)',
                           showarrow=False,
                           xanchor='left', yanchor='top',
                           font=dict(size=10, color='gray'))
    return fig

## Raw hourly time series (full dataset)

Shaded region = sensor was in a different room; values are not comparable to post-move readings.

In [4]:
fig = px.line(df, y='radon', title='Hourly radon (pCi/L) — full dataset',
              labels={'radon': 'pCi/L', 'ts': ''})
fig.update_traces(line=dict(width=1))
annotate(fig, events=EVENTS, shade_pre_move=True)
fig.show()

---

# Post-sensor-move analysis

Everything below uses **`df_post`** — readings from 2026-05-14 onward only.

In [5]:
df_post['radon'].describe().round(2)

count    346.00
mean       1.24
std        1.54
min        0.20
25%        0.60
50%        0.70
75%        1.20
max       15.00
Name: radon, dtype: float64

## Daily distribution (percentiles)

For each day: the **median (p50)** line shows the typical level; the **IQR band (p25–p75)** shows the middle half of readings; **p90/p95** capture the upper tail; the **daily max** marker is the single worst hour. A day where median is low but max hits EPA is a day where the system is mostly working but failed at least once.

In [6]:
daily = pd.DataFrame({
    'p25':       df_post['radon'].resample('D').quantile(0.25),
    'p50':       df_post['radon'].resample('D').quantile(0.50),
    'p75':       df_post['radon'].resample('D').quantile(0.75),
    'p90':       df_post['radon'].resample('D').quantile(0.90),
    'p95':       df_post['radon'].resample('D').quantile(0.95),
    'daily_max': df_post['radon'].resample('D').max(),
})

fig = go.Figure()
# IQR band (p25-p75) — invisible upper bound, then fill down from p25
fig.add_trace(go.Scatter(x=daily.index, y=daily['p75'],
                         line=dict(color='rgba(0,0,0,0)'), mode='lines',
                         showlegend=False, hoverinfo='skip'))
fig.add_trace(go.Scatter(x=daily.index, y=daily['p25'], name='IQR (p25–p75)',
                         line=dict(color='rgba(0,0,0,0)'), mode='lines',
                         fill='tonexty', fillcolor='rgba(70, 130, 180, 0.20)'))
# percentile lines
fig.add_trace(go.Scatter(x=daily.index, y=daily['p50'], name='p50 (median)',
                         line=dict(color='steelblue', width=2.5)))
fig.add_trace(go.Scatter(x=daily.index, y=daily['p90'], name='p90',
                         line=dict(color='goldenrod', width=2)))
fig.add_trace(go.Scatter(x=daily.index, y=daily['p95'], name='p95',
                         line=dict(color='darkorange', width=2)))
fig.add_trace(go.Scatter(x=daily.index, y=daily['daily_max'], name='daily max',
                         mode='markers',
                         marker=dict(color='crimson', size=9, symbol='diamond')))
fig.update_layout(title='Daily radon distribution: median, IQR, upper percentiles, and max',
                  yaxis_title='pCi/L', xaxis_title='')
annotate(fig, events=EVENTS_POST)
fig.show()

## 24-hour rolling: mean vs. p90

The 24-hour rolling mean (blue) shows the typical level over the prior day. The 24-hour rolling 90th percentile (red, dashed) shows what level the worst 10% of hours in that window hit. The gap between them is informative: a low rolling mean alongside a high rolling p90 indicates the system is "mostly working but occasionally failing" — a failure mode that a mean-only chart hides.

Hourly readings are shown in light gray for context. The 7-day window is omitted here because the post-move dataset spans less than 3 weeks; daily-level smoothing is already covered in the daily-distribution chart above.

In [7]:
window_24h = df_post['radon'].rolling('24h')

ma = pd.DataFrame({
    'hourly':  df_post['radon'],
    '24h mean': window_24h.mean(),
    '24h p90':  window_24h.quantile(0.9),
})

fig = px.line(ma, title='24-hour rolling: mean vs. p90',
              labels={'value': 'pCi/L', 'ts': '', 'variable': ''})
fig.update_traces(selector=dict(name='hourly'),   line=dict(color='lightgray', width=1))
fig.update_traces(selector=dict(name='24h mean'), line=dict(color='steelblue', width=2.5))
fig.update_traces(selector=dict(name='24h p90'),  line=dict(color='crimson', width=2.5, dash='dash'))
annotate(fig, events=EVENTS_POST)
fig.show()

## Hour-of-day pattern

Radon often peaks overnight (less air exchange) and dips during the day. If mitigation is effective and running 24/7, you'd expect this pattern to flatten.

In [8]:
by_hour = df_post.copy()
by_hour['hour'] = by_hour.index.hour

fig = px.box(by_hour, x='hour', y='radon', points=False,
             title='Radon distribution by hour of day (post-sensor-move)',
             labels={'radon': 'pCi/L', 'hour': 'hour of day'})
annotate(fig)
fig.show()

## Evening-only timeline (19:00–22:00)

The hour-of-day boxplot above shows readings peak in the evening. This filters the dataset down to just those hours so the temporal pattern is visible across phases: how does the evening profile in P1 (early mitigation), P2 (after the May 18 sealing + 3.5" WC change), and P3 (after the May 23 external adjustments) compare? Each dot is one evening hourly reading.

In [9]:
evening = df_post[df_post.index.hour.isin([19, 20, 21, 22])].copy()

fig = px.scatter(evening, y='radon',
                 title='Evening hours only (19:00–22:00) — the spike risk window',
                 labels={'radon': 'pCi/L', 'ts': ''})
fig.update_traces(marker=dict(size=8, color='steelblue', opacity=0.7))
annotate(fig, events=EVENTS_POST)
fig.show()

## Day-of-week pattern

In [10]:
by_dow = df_post.copy()
by_dow['dow'] = by_dow.index.day_name()
dow_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']

fig = px.box(by_dow, x='dow', y='radon', points=False,
             category_orders={'dow': dow_order},
             title='Radon distribution by day of week (post-sensor-move)',
             labels={'radon': 'pCi/L', 'dow': ''})
annotate(fig)
fig.show()

## Time spent in each radon band

How much of the post-move period was radon below WHO, between WHO and EPA, or at/above EPA? This is the single-bar summary; the next chart breaks it down day-by-day.

In [11]:
total = len(df_post)
bands = [
    ('< WHO (< 2.7)',     int((df_post['radon'] < WHO_LEVEL).sum()),                                                'steelblue'),
    ('WHO–EPA (2.7–4.0)', int(((df_post['radon'] >= WHO_LEVEL) & (df_post['radon'] < EPA_LEVEL)).sum()),            'goldenrod'),
    ('≥ EPA (≥ 4.0)',     int((df_post['radon'] >= EPA_LEVEL).sum()),                                               'crimson'),
]

fig = go.Figure()
for label, count, color in bands:
    pct = count / total * 100
    fig.add_trace(go.Bar(
        y=['post-move hours'], x=[count],
        name=label, orientation='h', marker_color=color,
        text=f'{count} h<br>({pct:.1f}%)' if pct >= 1 else f'{count}h',
        textposition='inside',
        textfont=dict(color='white', size=13),
        insidetextanchor='middle',
        hovertemplate=f'{label}<br>{count} hours<br>{pct:.1f}%<extra></extra>',
    ))

fig.update_layout(
    title=f'Time spent in each radon band  —  {total} post-move hourly readings',
    barmode='stack', xaxis_title='hours', yaxis_title='', height=320,
    legend=dict(orientation='h', y=-0.5, x=0),
    margin=dict(t=60, b=80, l=120, r=20),
)
fig.show()

## Hours per day above thresholds

Hours per day at or above each threshold. Any bar > 0 on the EPA series is a day where the mitigation system did not keep radon below the action level for the full day.

In [12]:
hours_above_who_daily = (df_post['radon'] >= WHO_LEVEL).resample('D').sum()
hours_above_epa_daily = (df_post['radon'] >= EPA_LEVEL).resample('D').sum()

fig = go.Figure()
fig.add_trace(go.Bar(x=hours_above_who_daily.index, y=hours_above_who_daily.values,
                     name=f'≥ WHO ({WHO_LEVEL})', marker_color='orange', opacity=0.55))
fig.add_trace(go.Bar(x=hours_above_epa_daily.index, y=hours_above_epa_daily.values,
                     name=f'≥ EPA ({EPA_LEVEL})', marker_color='crimson'))
fig.update_layout(title='Hours per day above radon thresholds',
                  yaxis_title='hours per day', xaxis_title='', barmode='overlay')
annotate(fig, levels=False, events=EVENTS_POST)
fig.show()

## Spike events (consecutive hours ≥ EPA)

Each row is a distinct excursion where radon stayed at or above the EPA action level (4.0 pCi/L) for one or more consecutive hours. Each event is a discrete occurrence that the mitigation system did not prevent — they do not average out into the mean. Pay attention to *when* the events occurred relative to the mitigation phases, not just *how many* there are. (Note: per the data-quality notes above, peaks shown here may be underestimates due to window-opening during real-time observations.)

In [13]:
above = df_post['radon'] >= EPA_LEVEL
groups = (above != above.shift()).cumsum()
spike_events = []
for _, sub in df_post[above].groupby(groups[above]):
    spike_events.append({
        'start':  sub.index.min(),
        'end':    sub.index.max(),
        'hours':  len(sub),
        'peak':   sub['radon'].max(),
        'mean':   round(sub['radon'].mean(), 2),
    })
spike_events_df = pd.DataFrame(spike_events)
print(f'{len(spike_events_df)} distinct events >= EPA across {len(df_post)} hours ({len(df_post)/24:.1f} days) of post-move data')
spike_events_df

10 distinct events >= EPA across 346 hours (14.4 days) of post-move data


,start,end,hours,peak,mean
0,2026-05-14 20:29:00,2026-05-14 21:29:00,2,7.5,6.40
1,2026-05-15 10:29:00,2026-05-15 10:29:00,1,4.1,4.10
2,2026-05-15 13:29:00,2026-05-15 14:29:00,2,6.5,5.35
3,2026-05-16 12:29:00,2026-05-16 12:29:00,1,5.3,5.30
4,2026-05-16 21:29:00,2026-05-16 23:29:00,3,15.0,10.07
5,2026-05-17 10:29:00,2026-05-17 10:29:00,1,4.7,4.70
6,2026-05-24 23:31:00,2026-05-24 23:31:00,1,4.2,4.20
7,2026-05-25 20:31:00,2026-05-25 21:31:00,2,6.6,6.30
8,2026-05-26 20:31:00,2026-05-26 21:31:00,2,6.3,5.90
9,2026-06-02 19:31:00,2026-06-02 20:31:00,2,10.3,8.45


## Phase comparison

Three mitigation phases bounded by the May 18 and May 23 events. Phases are short (especially P1 at ~4 days), so treat differences as suggestive rather than conclusive.

In [14]:
def phase_stats(s):
    if len(s) == 0:
        return pd.Series({'days': 0, 'n': 0, 'mean': float('nan'),
                          'p50': float('nan'), 'p75': float('nan'),
                          'p90': float('nan'), 'p95': float('nan'),
                          'max': float('nan'),
                          'above_EPA': '0h (0.0%)',
                          'above_WHO': '0h (0.0%)'})
    span_days = (s.index.max() - s.index.min()).total_seconds() / 86400
    n_epa = int((s >= EPA_LEVEL).sum())
    n_who = int((s >= WHO_LEVEL).sum())
    return pd.Series({
        'days':       round(span_days, 1),
        'n':          len(s),
        'mean':       round(s.mean(), 2),
        'p50':        round(s.quantile(0.50), 2),
        'p75':        round(s.quantile(0.75), 2),
        'p90':        round(s.quantile(0.90), 2),
        'p95':        round(s.quantile(0.95), 2),
        'max':        round(s.max(), 2),
        'above_EPA':  f'{n_epa}h ({n_epa / len(s) * 100:.1f}%)',
        'above_WHO':  f'{n_who}h ({n_who / len(s) * 100:.1f}%)',
    })

phase_series = {}
for name, start, end in PHASES:
    start_ts = pd.Timestamp(start)
    end_ts   = pd.Timestamp(end) if end else (df.index.max() + pd.Timedelta(seconds=1))
    mask = (df.index >= start_ts) & (df.index < end_ts)
    phase_series[name] = df.loc[mask, 'radon']

summary = pd.DataFrame({name: phase_stats(s) for name, s in phase_series.items()}).T
summary

,days,n,mean,p50,p75,p90,p95,max,above_EPA,above_WHO
"P1: fan + 2.5"" WC",4.0,70,2.16,1.3,2.82,4.76,6.72,15.0,10h (14.3%),18h (25.7%)
"P2: 3.5"" WC + sealing",5.0,89,0.88,0.8,1.0,1.42,1.86,3.8,0h (0.0%),2h (2.2%)
P3: + ext. tweaks,10.4,187,1.07,0.7,1.0,2.34,3.37,10.3,7h (3.7%),14h (7.5%)


In [15]:
phase_long = pd.concat([
    s.to_frame('radon').assign(phase=name)
    for name, s in phase_series.items()
]).reset_index()

fig = px.box(phase_long, x='phase', y='radon', points='outliers',
             title='Radon distribution by mitigation phase',
             labels={'radon': 'pCi/L', 'phase': ''})
annotate(fig)
fig.show()

## Observations and remaining questions

**1. Seasonality — winter not yet observed.** Radon levels in this climate are typically highest in winter (Nov–Mar) due to closed-window operation and the stack effect actively pulling soil gas through the foundation. The data above covers late spring only, which represents a seasonal floor. Effective year-round performance can only be confirmed by observing the system through at least one full heating season.

**2. Window-opening during elevated readings.** Per the data-quality note in the introduction, when elevated readings were observed in real time, windows were opened to ventilate. This likely lowered some readings during those events, so the spike peaks and counts above are likely underestimates of the unmitigated indoor levels.

**3. Mean vs. distribution.** EPA's 4.0 pCi/L is an annualized-average benchmark, but a mitigation system's mechanical function is to prevent individual excursions of soil gas. A system whose mean is below 4.0 but whose 95th percentile or daily max still hits EPA is reducing typical exposure without fully controlling source-gas entry. Distribution metrics (percentiles, hours-above-threshold, individual spike events) belong alongside the mean when judging effectiveness.

**4. Spike events after the May 23 adjustments.** Multiple EPA-exceeding events occurred during P3 (the post-external-tweak period), including a 2-hour event peaking at 10.3 pCi/L on June 2 — the latest reading in the dataset. The spike-events table above lists each one with timestamps and durations.

**5. Evening risk window.** Spike events cluster between 19:00 and 22:00. The hour-of-day boxplot, evening-only timeline, and individual spike-event timestamps all converge on this pattern. The cause is not yet identified — candidate hypotheses include HVAC cycles, kitchen exhaust depressurization, or evening interior cool-down creating negative pressure at the foundation. Worth investigating before declaring the system stable.